<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    04 · Quimioinformatica y RDKit
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:620px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 2 — Sin experiencia previa en programación</em>
  </p>
</div>


# Introducción a la quimioinformática usando RDKit

---
### En esta lección aprenderás:

- cómo leer SMILES con `rdkit`.
- cómo manipular y visualizar moléculas.
- cómo calcular descriptores moleculares.
- cómo calcular la similitud entre moléculas usando fingerprints.

---

El notebook de hoy es parte de una serie de notebooks de quimioinformática que acompañan el curso **Ciencia de Datos en Descubrimiento de Fármacos**.
El fármaco de ejemplo es el **sorafenib**, un inhibidor de quinasas aprobado para el tratamiento de varios tipos de cáncer (carcinoma hepatocelular, carcinoma de células renales, cáncer de tiroides diferenciado).

**Target:** RAF quinasa y receptores de factores de crecimiento (VEGFR, PDGFR)

---

In [ ]:
sorafenib = " " # write the smiles in " " to tell Python that it's a string
print(sorafenib)
type(sorafenib)

Como puedes ver, los SMILES se almacenan como `str` (`string`). En realidad podemos manipular este `string` y también aplicarle funciones. Sin embargo, el problema es que aunque Python entiende los SMILES como un `string`, no puede inferir la estructura molecular subyacente a partir de ellos. No tenemos forma de obtener información sobre el peso molecular, la carga, la aromaticidad, etc. solo a partir del `string`.

Para este fin se necesita `rdkit`, una librería de quimioinformática de código abierto para Python:

In [ ]:
# Installs RDKit
!pip install rdkit==2022.3.4

In [ ]:
from rdkit.Chem import AllChem as Chem
from rdkit.Chem.Draw import IPythonConsole

sorafenib = Chem.MolFromSmiles(sorafenib)
sorafenib

Con la ayuda de RDKit, los SMILES pueden leerse y representarse como una molécula válida.
El `type(sorafenib)` ahora es:

In [ ]:
type(sorafenib)

La función `Chem.MolFromSmiles` convierte el string SMILES en un nuevo tipo de variable: el **RDKit-Mol**. Mientras una molécula esté almacenada como `rdkit.Chem.rdchem.Mol` en Python, puedes aplicarle todas las funciones de rdkit. También puedes usar `Chem.MolToSmiles(mol)` para obtener la molécula en formato SMILES de nuevo.

Observa lo que ocurre cuando introduces un SMILES inválido:

In [ ]:
Chem.MolToSmiles(sorafenib)

El `string` SMILES del sorafenib ahora se ve diferente al que leíste originalmente. La diferencia está en la representación de los anillos aromáticos. En el string original se usaban explícitamente dobles enlaces `=`, pero ahora las `C` mayúsculas con doble enlace son reemplazadas por `c` minúsculas. RDKit canonicaliza automáticamente los SMILES. Esto significa que independientemente del formato en que introduzcas el SMILES, siempre obtendrás la misma representación canónica. Esto es muy útil para eliminar duplicados en bases de datos.

In [ ]:
Chem.MolFromSmiles('CNC(=[O-])c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1') # (=[O-]) instead of (=O)

### RDKit
Ahora que tienes el sorafenib en el formato correcto, también puedes obtener información sobre esta molécula:

In [ ]:
sorafenib.GetNumAtoms() # How many atoms does sorafenib consist of?

Existen varias funciones que se pueden usar para obtener información sobre las moléculas. `rdkit` asigna un índice a cada átomo y enlace. Con este índice puedes seleccionar átomos o enlaces individuales. Puedes ver qué índice tiene cada átomo cambiando las opciones de `Draw` de la siguiente manera:

In [ ]:
from rdkit.Chem import Draw 
IPythonConsole.drawOptions.addAtomIndices = True 
IPythonConsole.drawOptions.addBondIndices = False
IPythonConsole.molSize = (500, 500) 

In [ ]:
sorafenib

Los átomos individuales pueden seleccionarse por su índice con `.GetAtomWithIdx()`. Otras funciones permiten obtener más información sobre cada átomo:

In [ ]:
print("Symbol of atom with index 3")
print(sorafenib.GetAtomWithIdx(3).GetSymbol())

print("\nMass of atom with index 3")
print(sorafenib.GetAtomWithIdx(3).GetMass())

print("\nHybridization of atom with 3")
print(sorafenib.GetAtomWithIdx(3).GetHybridization())


Con la función `.SetAtomicNum()` también puedes cambiar átomos individuales y, por ejemplo, convertir la cetona en una imina.

In [ ]:
sorafenib.GetAtomWithIdx(3).SetAtomicNum(7)
display(sorafenib)
print(Chem.MolToSmiles(sorafenib))
sorafenib.GetAtomWithIdx(3).SetAtomicNum(8) # Change is reversed again  

¿Puedes reemplazar uno de los átomos de flúor por un átomo de carbono?

In [ ]:
sorafenib._____.______ # write your solution here

display(sorafenib)
print(Chem.MolToSmiles(sorafenib))
sorafenib = Chem.MolFromSmiles("CNC(=O)C1=NC=CC(=C1)OC2=CC=C(C=C2)NC(=O)NC3=CC(=C(C=C3)Cl)C(F)(F)F")

<details>
<summary><strong>Solución:</strong></summary>

```python
    sorafenib.GetAtomWithIdx(31).SetAtomicNum(6)
```
</details>

También se pueden usar funciones similares para los enlaces. A cada enlace también se le asigna un índice.

In [ ]:
IPythonConsole.drawOptions.addAtomIndices = False # don't show atom indices
IPythonConsole.drawOptions.addBondIndices = True # show bond indices

sorafenib

In [ ]:
print("Which type of bond is bond 4")
print(sorafenib.GetBondWithIdx(4).GetBondType())

print("\nIs bond 4 in a ring of size 7")
print(sorafenib.GetBondWithIdx(4).IsInRingSize(7))

print("\nIs bond 4 in a ring of size 6")
print(sorafenib.GetBondWithIdx(4).IsInRingSize(6))

IPythonConsole.drawOptions.addBondIndices = False

### Descriptores

Más útil que la información sobre átomos individuales son los descriptores calculados para una molécula completa. Con diferentes submódulos de `rdkit` puedes calcular distintas propiedades de las moléculas:

In [ ]:
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem.Crippen import MolLogP

print("LogP",MolLogP(sorafenib))
print("Molecular Weight",MolWt(sorafenib))

## Alternativas al Sorafenib

El objetivo es encontrar moléculas alternativas al sorafenib. Ya se ha realizado una preselección. Los SMILES se encuentran en la lista `smiles`.

In [ ]:
smiles = [
    "CNC(=O)c1cc(Oc2ccc(NC(=S)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "C[C@@H](NC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1)C(=O)NO",
    "CNC(=O)c1cc(Oc2ccc(NC(=S)Nc3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc2)ccn1",
    "N#Cc1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "CN(C)c1ccc(NC(=O)c2cc(Oc3ccc(NC(=O)Nc4ccc(Cl)c(C(F)(F)F)c4)cc3)ccn2)cc1", 
    "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Br)c(C(F)(F)F)c3)cc2)ccn1",
    "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(OC(F)(F)F)cc3)cc2)ccn1",
    "CCNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3cccc(C(F)(F)F)c3)cc2)ccn1"
]

Para evitar tener que convertir cada SMILES individualmente en un objeto `mol`, escribe un `for loop`.

Puedes mostrar múltiples moléculas una al lado de la otra con la función `Draw.MolsToGridImage(mols)`.

In [ ]:
mols = [ _____ for x in ______] # write your solution here
Draw.MolsToGridImage(mols,subImgSize=(300, 300)) #subImgSize allows to show larger images

<details>
<summary><strong>Solución:</strong></summary>

```python
mols = [Chem.MolFromSmiles(x) for x in smiles]
Draw.MolsToGridImage(mols,subImgSize=(300, 300))
```
</details>

Para evitar costos innecesarios, debes seleccionar únicamente las moléculas más prometedoras. Para esto, puedes aplicar lo que has aprendido hasta ahora.
Una regla de oro simple pero importante en el desarrollo de fármacos es la ["Regla de los Cinco de Lipinski"](https://en.wikipedia.org/wiki/Lipinski%27s_rule_of_five). Establece que las moléculas con buena biodisponibilidad oral suelen cumplir con los siguientes cuatro criterios:

- Peso molecular ≤ 500 Da
- LogP ≤ 5 (lipofilia)
- Número de donadores de puentes de hidrógeno ≤ 5
- Número de aceptores de puentes de hidrógeno ≤ 10

Una molécula que viola más de una de estas reglas puede tener problemas de biodisponibilidad oral. Sin embargo, hay excepciones notables como la ciclosporina A.

Ya calculaste el valor de LogP y el peso molecular con funciones de `rdkit`.
El submódulo `Lipinski` en RDKit ofrece más funciones, entre ellas para el cálculo del número de donadores y aceptores de puentes de hidrógeno.

Primero calcula el número de donadores de hidrógeno (`NumHDonors()`) para cada molécula en `mols`:

In [ ]:
from rdkit.Chem.Lipinski import NumHAcceptors, NumHDonors

NumDonors = [______(x) for x in ______] # write your solution here

<details>
<summary><strong>Solución:</strong></summary>

```python
NumDonors = [NumHDonors(x) for x in mols]
```
</details>

Para mostrar el número de donadores junto con las moléculas puedes usar la función `MolsToGridImage()`. Aquí simplemente tienes que pasar `NumDonors` a la variable de entrada `legends`. El problema es que la función siempre espera `Strings`, no `integers`. Por eso usamos otro `for-loop` para convertir los `integers` en `strings`:

In [ ]:
NumDonors = [str(x) for x in NumDonors] # convert the list of ints to a list of strings
Draw.MolsToGridImage(mols, legends = NumDonors,subImgSize = (300,300))

En el pie de imagen puedes ver el número de donadores de puentes de hidrógeno. Todas las moléculas tienen menos donadores que el límite máximo establecido por Lipinski. Podemos repetir lo mismo para los aceptores.
Sin embargo, esta vez escribe el `for-loop` de manera que los `integers` se conviertan inmediatamente en `strings`. Así evitas tener que hacer la conversión en un paso separado:

In [ ]:
NumAcceptors = [str(________) for x in ________] # write your solution here
Draw.MolsToGridImage(mols, legends = NumAcceptors, subImgSize= (300,300))

<details>
<summary><strong>Solución:</strong></summary>

```python
NumAcceptors = [str(NumHAcceptors(x)) for x in mols]
Draw.MolsToGridImage(mols, legends = NumAcceptors, subImgSize= (300,300))
```
</details>

Ninguna de las moléculas viola la regla de Lipinski tampoco para los aceptores.
Ahora calcula el peso molecular (`MolWt()`) para las alternativas al sorafenib.

In [ ]:
molWeight = [str(_________) for __ in _______] # write your solution here
Draw.MolsToGridImage(mols, legends = molWeight, subImgSize=(300, 300))

<details>
<summary><strong>Solución:</strong></summary>

```python
molWeight = [str(MolWt(x)) for x in mols]
Draw.MolsToGridImage(mols, legends = molWeight, subImgSize=(300, 300))
```
</details>

Algunas moléculas son en realidad más pesadas de lo que permite la "Regla de los Cinco de Lipinski".

Lo último que harás es calcular el LogP (`MolLogP()`).

In [ ]:
logP = [_______ for ____ in _____] # write your solution here, remember the str() function
Draw.MolsToGridImage(mols, legends = logP,subImgSize=(300, 300))

<details>
<summary><strong>Solución:</strong></summary>

```python
logP = [str(MolLogP(x)) for x in mols]
Draw.MolsToGridImage(mols, legends = logP,subImgSize=(300, 300))
```
</details>

De hecho, la mayoría de las moléculas supera el valor de LogP de Lipinski. Solo tres moléculas tienen un valor menor a cinco. Estas últimas dos moléculas son las únicas que cumplen con las cuatro reglas de Lipinski. Por lo tanto, pueden ser especialmente adecuadas como fármacos. Sin embargo, no se deben descartar todas las demás moléculas solo por eso — la Regla de los Cinco de Lipinski es solo una guía y hay excepciones importantes.

Con ayuda de `numpy` podemos crear un `array` booleano que muestre qué moléculas cumplen con la condición de LogP < 5:

In [ ]:
import numpy as np

logP = [MolLogP(x) for x in mols]
logP = np.array(logP) # the list has to be converted to an array
logP < 5.0

Ahora haz lo mismo para el peso molecular (`MolWt()`).

In [ ]:
molWeight = [____() ___ ____ ___ ___] # write your solution here
molWeight = ______________ # convert the list to an array
molWeight < 500

<details>
<summary><strong>Solución:</strong></summary>

```python
molWeight = [MolWt(x) for x in mols]
molWeight = np.array(molWeight)
molWeight < 500
```
</details>

Para seleccionar las moléculas que tienen un peso inferior a 500 **o** un LogP inferior a cinco, puedes usar el símbolo `|`. El `|` representa "o" (OR lógico). La expresión `(logP < 5) | (molWeight < 500)` dará `True` para los elementos que cumplan al menos una de las dos condiciones. Se dará `False` si un elemento no cumple ninguna condición.

In [ ]:
(logP < 5) | (molWeight<500)

Con este array de booleanos (`bool`) ahora podemos seleccionar las moléculas que pasan el filtro.

In [ ]:
mols=np.array(mols)
mols_subset=mols[(logP < 5) | (molWeight<500)] # we convert the mol-list to an array
Draw.MolsToGridImage(mols_subset, subImgSize=(300, 300))

Calculando descriptores, puedes reducir el número de moléculas candidatas. En el siguiente paso aprenderás cómo reducir aún más la selección con una búsqueda de similitud.

## Fingerprints y Búsqueda de Similitud

RDKit también puede calcular varios *fingerprints moleculares*. Entre ellos se encuentra el *Extended Connectivity Fingerprint* (ECFP), desarrollado originalmente por [Hahn et al.](https://pubs.acs.org/doi/10.1021/ci100050t) en 2010.
RDKit tiene una versión modificada que llaman *Morgan Fingerprint*. El parámetro más importante es el `radio`. Con un radio de 2, el fingerprint se denomina ECFP4, y con un radio de 3, ECFP6. Cuanto mayor sea el radio, más información sobre el entorno de cada átomo se incorpora al fingerprint.

La similitud entre dos fingerprints se calcula generalmente con la **Similitud de Tanimoto**. Esta métrica va de 0 (ninguna similitud) a 1 (moléculas idénticas). En la práctica, se considera que dos moléculas son similares si su similitud de Tanimoto es ≥ 0.8.

Primero calculamos el fingerprint para el sorafenib:

In [ ]:
from rdkit import DataStructs
fp_sorafenib = Chem.GetMorganFingerprint(sorafenib,radius=2)
fp_sorafenib

El Morgan fingerprint no se almacena como un `np.array` regular. En los siguientes notebooks aprenderás cómo obtener también los vectores *normales* de los fingerprints.

Has calculado el fingerprint para el sorafenib, pero para calcular la similitud necesitas también los fingerprints de las otras moléculas. Escribe un `for-loop` para calcular el fingerprint de cada molécula en `mols_subset`:

In [ ]:
fp_mols = [Chem.GetMorganFingerprint( ___ ,radius = 2) for __ in ___ ]

<details>
<summary><strong>Solución:</strong></summary>

```python
    fp_mols = [Chem.GetMorganFingerprint( x ,radius = 2) for x in mols_subset]
```
</details>
<br>
Para calcular la similitud, usa la función descrita anteriormente `TanimotoSimilarity(fp1, fp2)`

In [ ]:
DataStructs.TanimotoSimilarity(fp_sorafenib,fp_mols[5])

Escribe un `for-loop` que calcule la similitud con el sorafenib para cada molécula en `fp_mols`.

In [ ]:
sorafenib_similarity = [DataStructs.TanimotoSimilarity(___ , ___ ) for x in ____]
sorafenib_similarity

<details>
<summary><strong>Solución:</strong></summary>

```python
    sorafenib_similarity=[DataStructs.TanimotoSimilarity(fp_sorafenib, x) for x in fp_mols]

```
</details>
<br>


In [ ]:
Draw.MolsToGridImage(mols_subset,legends = [str(x) for x in sorafenib_similarity],subImgSize=(300, 300))

Arriba puedes ver la similitud de cada molécula con el sorafenib. Una regla de oro comúnmente usada es que las moléculas con una similitud de 0.8 o mayor son suficientemente similares para considerarse una alternativa relevante. En nuestro caso, esto significa que solo probaríamos una molécula. La molécula con una similitud de 1.0 es idéntica al sorafenib — solo se representa de forma diferente.

## Ejercicio Práctico: Alternativas para el Antibiótico Norfloxacino

Como tarea algo más desafiante, ahora debes aplicar lo que has aprendido de forma independiente.
Básicamente, la tarea es muy similar a las anteriores, pero recibirás menos ayuda.
Primero, busca el `string` SMILES del **norfloxacino** (puedes buscarlo en PubChem o ChEMBL). A continuación, conviértelo en una molécula RDKit y muéstrala.

In [ ]:
# If you do this task at a later time, you can use this cell
# to import the required libraries at once
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from rdkit.Chem.Descriptors import MolWt 
from rdkit.Chem.Crippen import MolLogP
from rdkit.Chem.Lipinski import NumHAcceptors, NumHDonors
from rdkit import DataStructs

In [ ]:
norfloxacin = "CCN1C=C(C(=O)C2=CC(=C(C=C21)N3CCNCC3)F)C(=O)O"
# convert the string to mol
norfloxacin = 
# display the molecule
norfloxacin

A continuación, calcula los descriptores del norfloxacino que son importantes para la *Regla de los Cinco de Lipinski*.

In [ ]:
# Calcula el PM (peso molecular)
MW_norfloxacin = 

# Calcula el número de aceptores de puente de H
NumHAcceptors_norfloxacin = 

# Calcula el número de donadores de puente de H
NumHDonors_norfloxacin = 

# Calcula el logP
logP_norfloxacin = 

La siguiente celda muestra los descriptores calculados:

In [ ]:
print("MW:", MW_norfloxacin)
print("NumHAcceptors", NumHAcceptors_norfloxacin)
print("NumHDonors", NumHDonors_norfloxacin)
print("LogP",logP_norfloxacin)

En la siguiente celda se listan posibles alternativas al norfloxacino. Convierte los SMILES al formato `mol` y luego calcula los descriptores. La forma más sencilla es usar la notación `Lista = [función(x) for x in OtraLista]` utilizada varias veces anteriormente.

In [ ]:
# Don't forget to run this cell.
quinolones = ["C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O",
             "CN1CCN(CC1)C2=C(C=C3C(=C2F)N(C=C(C3=O)C(=O)O)CCF)F",
             "CCN1C=C(C(=O)C2=CC(=C(C(=C21)F)N3CCNC(C3)C)F)C(=O)O",
             "CC1CCC2=C3N1C=C(C(=O)C3=CC(=C2N4CCC(CC4)O)F)C(=O)O",
             "CC1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)F)C(=O)O"
             "CCN1C=C(C(=O)C2=CC(=C(C=C21)N3CCN(CC3)C)F)C(=O)O",
             "CN1CCN(CC1)C2=C(C=C3C4=C2SCCN4C=C(C3=O)C(=O)O)F",
             "CCN1C=C(C(=O)C2=CC(=C(N=C21)N3CCNCC3)F)C(=O)O",
             "CNC1CCCN(C1)C2=C(C=C3C(=C2OC)N(C=C(C3=O)C(=O)O)C4CC4)F",
             "CC1CN(CCN1)C2=C(C(=C3C(=C2)N(C=C(C3=O)C(=O)O)C4CC4)C)F",
             "C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)F)C(=O)O",
             "C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2C4(CC4)N)F)C(=O)O",
             "C[C@@H]1CN(C[C@@H](N1)C)C2=C(C(=C3C(=C2F)N(C=C(C3=O)C(=O)O)C4CC4)N)F",
             "CC1CN(CCN1)C2=C(C=C3C(=C2)N(C=C(C3=O)C(=O)O)C4=C(C=C(C=C4)F)F)F",
             "C1CN(CC1N)C2=C(C=C3C(=O)C(=CN(C3=N2)C4=C(C=C(C=C4)F)F)C(=O)O)F"]

In [ ]:
# Convierte los strings a mol
quinolones = 

In [ ]:
# Calcula los cuatro descriptores para todas las moléculas de la lista
MW_quinolones = 
NumHAcceptors_quinolones = 
NumHDonors_quinolones = 
logP_quinolones = 

En la siguiente celda puedes visualizar las moléculas con los descriptores calculados. No es necesario que entiendas el código en detalle.

Debes leer los valores tú mismo. (`Ctrl` + rueda del ratón para hacer zoom.)

In [ ]:
legend = []
for i in range(len(MW_quinolones)):
    legend.append("MW: "+str(round(MW_quinolones[i]))+"\n"+
                 "NumHAcceptors: "+str(NumHAcceptors_quinolones[i])+"\n"+
                 "NumHDonors: "+str(NumHDonors_quinolones[i])+"\n"+
                 "logP: "+str(round(logP_quinolones[i], 4)))

Draw.MolsToGridImage(quinolones, molsPerRow=3, legends = legend,
                    subImgSize=(250,150), useSVG=True)

Como prácticamente todas las moléculas siguen la *Regla de los Cinco de Lipinski*, en este punto no descartaremos ninguna molécula y en cambio calcularemos directamente la similitud con el norfloxacino. Para ello, primero hay que calcular los fingerprints y luego la similitud de Tanimoto.

In [ ]:
# Primero calcula los fingerprints de las quinolonas y el norfloxacino.
norfloxacin_fp = 
quinolones_fp = 

Ahora calcula las similitudes de `quinolones` con el norfloxacino.

En la celda siguiente puedes visualizar las similitudes.

In [ ]:
quinolones_similarity = 

In [ ]:
Draw.MolsToGridImage(quinolones, legends = [str(round(x, 2)) for x in quinolones_similarity],
                    subImgSize=(250,200), useSVG=True)

Como puedes ver, la mayoría de las moléculas no son particularmente similares al norfloxacino (al menos según la Similitud de Tanimoto). Sin embargo, cada una de estas moléculas es en realidad un antibiótico de amplio espectro que estuvo disponible, al menos en el pasado. Por lo tanto, no se puede confiar únicamente en la similitud entre moléculas. La actividad biológica depende de muchos factores — y la Similitud de Tanimoto captura solo una parte de ellos.

---

**Resumen de lo aprendido en este notebook:**
- Representar moléculas como objetos RDKit desde SMILES
- Manipular átomos y enlaces individuales con sus índices
- Calcular descriptores fisicoquímicos: LogP, PM, donadores/aceptores de H
- Aplicar la Regla de los Cinco de Lipinski para filtrar moléculas
- Calcular fingerprints de Morgan y similitud de Tanimoto
- Hacer búsquedas de similitud para encontrar alternativas moleculares

*NB-03 · Ciencia de Datos en Descubrimiento de Fármacos · UNAL 2026*